# Estrategia Momentum (MSCI-like)

Implementacion del algoritmo descrito en el enunciado: momentum 12M y 6M con lag de 1 mes, normalizacion Z-score por mes, score compuesto, seleccion y pesos.


In [25]:
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 50)

In [26]:

BACKTEST_START = pd.Timestamp('2015-10-31')

# Se usa el dataset filtrado ya preparado en EDA
# (incluye historial suficiente para lag=1 con R_12 y R_6)
df = pd.read_pickle('sp500_history_filtered.pkl')
source_file = 'sp500_history_filtered.pkl'

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

required_history_start = BACKTEST_START - pd.offsets.MonthEnd(13)
if df['date'].min() > required_history_start:
    raise ValueError(
        f'Historico insuficiente para iniciar en {BACKTEST_START.date()}. '
        f'Se necesita al menos desde {required_history_start.date()} y el dataset arranca en {df["date"].min().date()}.'
    )

print('Fuente:', source_file)
print('Rango de fechas:', df['date'].min().date(), '->', df['date'].max().date())
print('BACKTEST_START:', BACKTEST_START.date())


Fuente: sp500_history_filtered.pkl
Rango de fechas: 2013-01-02 -> 2026-01-30
BACKTEST_START: 2015-10-31


Paso A: Retorno Acumulado con "Lag" de 1 Mes 
Se deben calcular dos variables de retorno para cada uno de los 10 activos (9 sectores + GLD) -utilizaremos retornos logarítmos-: 
1.  Momentum 12 Meses (R_12): Rentabilidad desde el mes t-13 al mes t-1. 
2.  Momentum 6 Meses (R_6): Rentabilidad desde el mes t-7 al mes t-1. 
(Nota: El mes t-1 es el mes anterior al rebalanceo; se excluye el mes actual para 
evitar el ruido de la reversión a la media).

In [27]:
# Resample mensual: ultimo dia habil con precio disponible por simbolo
monthly = (
    df.set_index('date')
      .groupby('symbol')
      .resample('ME')
      .last()
      .reset_index()
)

# Retornos logaritmicos mensuales por simbolo
monthly['log_ret'] = np.log(monthly['close'] / monthly.groupby('symbol')['close'].shift(1))
monthly = monthly.dropna(subset=['log_ret'])
monthly.head()


,symbol,date,assetid,security_name,sector,industry,subsector,in_sp500,open,high,low,close,volume,unadjusted_close,log_ret
1,A,2013-02-28,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,26.761311,26.940660,26.549936,26.569151,5408812.0,41.480000,-0.076550
2,A,2013-03-31,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,27.030537,27.030537,26.657967,26.959877,3343916.0,41.970001,0.014599
3,A,2013-04-30,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,26.465258,26.760744,26.304668,26.619423,5373070.5,41.439999,-0.012709
4,A,2013-05-31,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,29.490776,29.850498,29.182444,29.195292,7175522.5,45.450001,0.092366
5,A,2013-06-30,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,27.698662,27.895128,27.524738,27.544064,8626299.0,42.759998,-0.058220


In [28]:
# Universo completo: todos los symbols del dataframe mensual
universe_ret = monthly[['date', 'sector', 'symbol', 'log_ret']].copy()
universe_ret['sector'] = universe_ret['sector'].fillna('Unknown')
universe_ret = universe_ret.sort_values(['symbol', 'date']).reset_index(drop=True)

print('Numero de symbols en el universo:', universe_ret['symbol'].nunique())
print('Rango del universo:', universe_ret['date'].min().date(), '->', universe_ret['date'].max().date())
universe_ret.head()

Numero de symbols en el universo: 858
Rango del universo: 2013-02-28 -> 2026-01-31


,date,sector,symbol,log_ret
0,2013-02-28,Health Care,A,-0.076550
1,2013-03-31,Health Care,A,0.014599
2,2013-04-30,Health Care,A,-0.012709
3,2013-05-31,Health Care,A,0.092366
4,2013-06-30,Health Care,A,-0.058220


In [30]:
# Mantener meses con al menos TOP_K activos elegibles
TOP_K = 20
assets_per_month = universe_ret.groupby('date')['symbol'].nunique()
valid_dates = assets_per_month[assets_per_month >= TOP_K].index

universe_ret = universe_ret[universe_ret['date'].isin(valid_dates)].copy()

print('Meses validos (>= 20 symbols):', len(valid_dates))
print('Primer mes valido:', valid_dates.min().date())
print('Ultimo mes valido:', valid_dates.max().date())
print('Min symbols/mes:', assets_per_month.loc[valid_dates].min())
print('Max symbols/mes:', assets_per_month.loc[valid_dates].max())

Meses validos (>= 20 symbols): 156
Primer mes valido: 2013-02-28
Ultimo mes valido: 2026-01-31
Min symbols/mes: 651
Max symbols/mes: 780


In [31]:
print('Universo listo para momentum. Filas:', len(universe_ret))
print('Symbols unicos:', universe_ret['symbol'].nunique())
universe_ret.head()

Universo listo para momentum. Filas: 111718
Symbols unicos: 858


,date,sector,symbol,log_ret
0,2013-02-28,Health Care,A,-0.076550
1,2013-03-31,Health Care,A,0.014599
2,2013-04-30,Health Care,A,-0.012709
3,2013-05-31,Health Care,A,0.092366
4,2013-06-30,Health Care,A,-0.058220


In [32]:
# Paso A: Momentum 12M y 6M con lag de 1 mes (se excluye el mes actual)
def add_momentum(df_in, window):
    col = f'R_{window}'
    df_in[col] = (
        df_in.groupby('symbol')['log_ret']
            .transform(lambda s: s.shift(1).rolling(window=window, min_periods=window).sum())
    )
    return df_in

mom = universe_ret.copy()
mom = add_momentum(mom, 12)
mom = add_momentum(mom, 6)
mom = mom.dropna(subset=['R_12', 'R_6']).copy()

# Backtest mensual desde fin de enero 2015
mom = mom[mom['date'] >= BACKTEST_START].copy()
rebalance_dates = pd.DatetimeIndex(sorted(mom['date'].unique()))

if BACKTEST_START not in rebalance_dates:
    first_available = rebalance_dates.min().date() if len(rebalance_dates) > 0 else 'N/A'
    raise ValueError(
        f'No hay datos suficientes para iniciar en {BACKTEST_START.date()} con lag de 1 mes y ventanas 12/6. '
        f'Primer mes disponible: {first_available}'
    )

expected_months = pd.date_range(start=BACKTEST_START, end=rebalance_dates.max(), freq='ME')
missing_months = expected_months.difference(rebalance_dates)
if len(missing_months) > 0:
    print('Advertencia: faltan meses de rebalanceo:', len(missing_months))

print('Primer rebalanceo:', rebalance_dates.min().date())
print('Ultimo rebalanceo:', rebalance_dates.max().date())
print('Numero de rebalanceos:', len(rebalance_dates))
mom.head()

Primer rebalanceo: 2015-10-31
Ultimo rebalanceo: 2026-01-31
Numero de rebalanceos: 124


,date,sector,symbol,log_ret,R_12,R_6
32,2015-10-31,Health Care,A,0.095231,-0.161025,-0.185479
33,2015-11-30,Health Care,A,0.102124,-0.035505,-0.085907
34,2015-12-31,Health Care,A,0.002492,-0.011456,0.020578
35,2016-01-31,Health Care,A,-0.104803,0.034063,0.086031
36,2016-02-29,Health Care,A,-0.008000,0.007407,-0.078389


Paso B: Normalización por Factor (Z-Score)

In [33]:
# Paso B: Z-score por mes
for col in ['R_12', 'R_6']:
    mu = mom.groupby('date')[col].transform('mean')
    sigma = mom.groupby('date')[col].transform('std').replace(0, np.nan)
    suffix = col.split('_')[1]
    mom[f'Z_{suffix}'] = (mom[col] - mu) / sigma

# Paso C: Score final
mom['score'] = (mom['Z_12'] + mom['Z_6']) / 2
mom = mom.dropna(subset=['score']).copy()
mom.head()

,date,sector,symbol,log_ret,R_12,R_6,Z_12,Z_6,score
32,2015-10-31,Health Care,A,0.095231,-0.161025,-0.185479,-0.284612,-0.250914,-0.267763
33,2015-11-30,Health Care,A,0.102124,-0.035505,-0.085907,-0.090601,-0.132216,-0.111409
34,2015-12-31,Health Care,A,0.002492,-0.011456,0.020578,0.042733,0.298579,0.170656
35,2016-01-31,Health Care,A,-0.104803,0.034063,0.086031,0.261674,0.573796,0.417735
36,2016-02-29,Health Care,A,-0.008000,0.007407,-0.078389,0.305596,0.246858,0.276227


Paso C: Puntuación Compuesta y Selección

In [34]:
# Seleccion Top-20 y pesos fijos del 5%
TOP_K = 20
FIXED_WEIGHT = 0.05

# Orden global por fecha y score para tomar los 20 mejores de cada mes
selection = (
    mom.sort_values(['date', 'score', 'symbol'], ascending=[True, False, True])
       .groupby('date', group_keys=False)
       .head(TOP_K)
       .copy()
)

# Ranking 1..20 por fecha
selection['rank'] = selection.groupby('date')['score'].rank(method='first', ascending=False).astype(int)
selection['weight'] = FIXED_WEIGHT
selection = selection[['date', 'rank', 'sector', 'symbol', 'score', 'weight']].copy()

# Validacion: exactamente 20 activos por fecha de rebalanceo
counts = selection.groupby('date')['symbol'].size()
invalid = counts[counts != TOP_K]
if len(invalid) > 0:
    raise ValueError(f'Hay fechas sin 20 activos seleccionados: {invalid.to_dict()}')

print('Activos seleccionados (primeras filas):')
print(selection.head(20))
print('Fechas de rebalanceo:', selection['date'].nunique())
print('Activos por fecha (min/max):', counts.min(), '/', counts.max())
selection.head()

Activos seleccionados (primeras filas):
             date  rank                  sector       symbol     score  weight
14319  2015-10-31     1             Industrials         BLDR  2.581801    0.05
896    2015-10-31     2             Health Care  ABMD-202212  2.419179    0.05
27415  2015-10-31     3  Consumer Discretionary          CZR  2.366657    0.05
26632  2015-10-31     4  Communication Services   CVC-201606  2.229772    0.05
71588  2015-10-31     5  Communication Services         NFLX  1.944029    0.05
32704  2015-10-31     6             Health Care         DXCM  1.855606    0.05
40582  2015-10-31     7             Industrials          FIX  1.681483    0.05
52200  2015-10-31     8             Health Care         INCY  1.650718    0.05
94824  2015-10-31     9               Utilities    TE-201606  1.487465    0.05
6234   2015-10-31    10  Consumer Discretionary         AMZN  1.481530    0.05
76767  2015-10-31    11             Industrials         PAYC  1.465530    0.05
4397   2015-

,date,rank,sector,symbol,score,weight
14319,2015-10-31,1,Industrials,BLDR,2.581801,0.05
896,2015-10-31,2,Health Care,ABMD-202212,2.419179,0.05
27415,2015-10-31,3,Consumer Discretionary,CZR,2.366657,0.05
26632,2015-10-31,4,Communication Services,CVC-201606,2.229772,0.05
71588,2015-10-31,5,Communication Services,NFLX,1.944029,0.05


In [35]:
# Open/Close mensuales por symbol
all_prices = monthly[['date', 'symbol', 'open', 'close']].copy()
all_prices['open'] = pd.to_numeric(all_prices['open'], errors='coerce')
all_prices['close'] = pd.to_numeric(all_prices['close'], errors='coerce')
all_prices = all_prices.dropna(subset=['open', 'close'])
all_prices = all_prices.sort_values(['symbol', 'date'])
all_prices.head()

,date,symbol,open,close
1,2013-02-28,A,26.761311,26.569151
2,2013-03-31,A,27.030537,26.959877
3,2013-04-30,A,26.465258,26.619423
4,2013-05-31,A,29.490776,29.195292
5,2013-06-30,A,27.698662,27.544064


In [36]:
# Anexar open/close a la seleccion
selection = selection.merge(
    all_prices,
    on=['date', 'symbol'],
    how='left',
)
selection.head()

,date,rank,sector,symbol,score,weight,open,close
0,2015-10-31,1,Industrials,BLDR,2.581801,0.05,11.670000,11.820000
1,2015-10-31,2,Health Care,ABMD-202212,2.419179,0.05,72.989998,73.660004
2,2015-10-31,3,Consumer Discretionary,CZR,2.366657,0.05,9.910000,9.900000
3,2015-10-31,4,Communication Services,CVC-201606,2.229772,0.05,32.560001,32.590000
4,2015-10-31,5,Communication Services,NFLX,1.944029,0.05,10.512000,10.838000


In [37]:
# Exportar CSV con los 20 activos seleccionados en cada fecha de rebalanceo
selection_out = selection[['date', 'rank', 'sector', 'symbol', 'score', 'weight', 'open', 'close']].copy()

counts_out = selection_out.groupby('date')['symbol'].size()
if (counts_out != 20).any():
    raise ValueError('El CSV no cumple 20 activos por fecha de rebalanceo.')

selection_out.to_csv('seleccion_momentum.csv', index=False)
print('CSV generado: seleccion_momentum.csv')
print('Filas:', len(selection_out))
print('Fechas:', selection_out['date'].nunique())
print('Activos por fecha (min/max):', counts_out.min(), '/', counts_out.max())


CSV generado: seleccion_momentum.csv
Filas: 2480
Fechas: 124
Activos por fecha (min/max): 20 / 20
